In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vit_timesformer_fcaf_probe import TimeSformer,TimeSformerFCAFLinearProbe
# from vivit import ViViTBackbone
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:1' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-25 01:24:22,277]::3493104939::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:1


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120

# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)

In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# TimeSformer FCAF Block-wise Linear Probe -- UDIVA / MSE
# ============================================================
#
# Probe targets:
#   block1 ... block12 : visual CLS after each complete TimeSformer block
#                        and BEFORE the final cross-attention
#   final_cross        : CLS after final cross-attention + final LayerNorm
#
# Backbone is completely frozen.
# Only independent probe heads are optimized.
#
# IMPORTANT:
#   Evaluation metric = MSE (lower is better).
#   DO NOT calculate/report 1-MSE.
#   O/C/E/A/N are also reported directly as MSE.
# ============================================================

PROBE_SEED = 42
PROBE_LR = 1e-3
WEIGHT_DECAY = 1e-4
PROBE_EPOCHS = 50
PATIENCE = 10

BACKBONE_CHECKPOINT = './save_timesformer_folder/model_best_multiseed.pth'


def set_probe_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def probe_seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


set_probe_seed(PROBE_SEED)

generator = torch.Generator()
generator.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(
    dataset=train_set_data,
    batch_size=batchsz,
    shuffle=True,
    num_workers=num_workerssz,
    worker_init_fn=probe_seed_worker,
    generator=generator
)

val_dataloader = DataLoader(
    dataset=val_set_data,
    batch_size=batchsz,
    shuffle=False,
    num_workers=num_workerssz,
    worker_init_fn=probe_seed_worker,
    generator=generator
)

# Re-create and load the trained FCAF backbone.
backbone = TimeSformer().to(device)

checkpoint = torch.load(BACKBONE_CHECKPOINT, map_location=device)
state_dict = (
    checkpoint["model"]
    if isinstance(checkpoint, dict) and "model" in checkpoint
    else checkpoint
)
backbone.load_state_dict(state_dict, strict=True)

# Frozen FCAF backbone + independent probe heads.
model = TimeSformerFCAFLinearProbe(
    backbone=backbone,
    dim=768,
    num_outputs=5
).to(device)

backbone_trainable = sum(
    p.numel() for p in model.backbone.parameters() if p.requires_grad
)
probe_trainable = sum(
    p.numel() for p in model.probe_heads.parameters() if p.requires_grad
)

print(f"Backbone trainable parameters: {backbone_trainable}")
print(f"Probe trainable parameters:    {probe_trainable}")
print(f"Metric: MSE | Patience: {PATIENCE}")
assert backbone_trainable == 0, "Backbone must remain frozen."

criterion = nn.MSELoss().to(device)

optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=PROBE_LR,
    weight_decay=WEIGHT_DECAY
)

probe_names = list(model.probe_heads.keys())

best_mse = {name: float("inf") for name in probe_names}
best_epoch = {name: -1 for name in probe_names}

best_mean_val_mse = float("inf")
no_improve_count = 0

for epoch in range(PROBE_EPOCHS):
    # -------------------- Train probe heads only --------------------
    model.train()

    train_sq_sum = {name: 0.0 for name in probe_names}
    train_count = 0

    for audiodata, fullshot, big_five_data in tqdm(
        train_dataloader,
        desc=f"Probe Train {epoch + 1}/{PROBE_EPOCHS}"
    ):
        fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
        audiodata = audiodata.to(device)
        big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)

        if big_five_data.ndim == 1:
            big_five_data = big_five_data.unsqueeze(0)

        optimizer.zero_grad()
        outputs = model(fullshot, audiodata)

        losses = {
            name: criterion(outputs[name], big_five_data)
            for name in probe_names
        }

        total_loss = sum(losses.values())
        total_loss.backward()
        optimizer.step()

        scalar_count = big_five_data.numel()
        train_count += scalar_count

        for name in probe_names:
            train_sq_sum[name] += torch.square(
                outputs[name].detach() - big_five_data
            ).sum().item()

    train_mse = {
        name: train_sq_sum[name] / train_count
        for name in probe_names
    }

    # -------------------- Validation --------------------
    model.eval()

    val_sq_sum = {name: 0.0 for name in probe_names}
    val_trait_sq_sum = {
        name: np.zeros(5, dtype=np.float64)
        for name in probe_names
    }
    val_count = 0
    val_sample_count = 0

    with torch.no_grad():
        for audiodata, fullshot, big_five_data in tqdm(
            val_dataloader,
            desc=f"Probe Valid {epoch + 1}/{PROBE_EPOCHS}"
        ):
            fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
            audiodata = audiodata.to(device)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)

            if big_five_data.ndim == 1:
                big_five_data = big_five_data.unsqueeze(0)

            outputs = model(fullshot, audiodata)

            val_sample_count += big_five_data.shape[0]
            val_count += big_five_data.numel()

            for name in probe_names:
                sq_err = torch.square(outputs[name] - big_five_data)
                val_sq_sum[name] += sq_err.sum().item()
                val_trait_sq_sum[name] += (
                    sq_err.sum(dim=0).detach().cpu().numpy()
                )

    val_mse = {
        name: val_sq_sum[name] / val_count
        for name in probe_names
    }

    val_trait_mse = {
        name: val_trait_sq_sum[name] / val_sample_count
        for name in probe_names
    }

    # Each representation independently tracks its lowest validation MSE.
    for name in probe_names:
        if val_mse[name] < best_mse[name]:
            best_mse[name] = val_mse[name]
            best_epoch[name] = epoch + 1

    # Global early stopping uses mean validation MSE across probe heads.
    mean_val_mse = float(np.mean([val_mse[name] for name in probe_names]))

    if mean_val_mse < best_mean_val_mse:
        best_mean_val_mse = mean_val_mse
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n" + "-" * 132)
    print(
        f"Epoch {epoch + 1:03d} | "
        f"Mean Val MSE={mean_val_mse:.6f} | "
        f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
        f"Patience={no_improve_count}/{PATIENCE}"
    )

    for name in probe_names:
        o_mse, c_mse, e_mse, a_mse, n_mse = val_trait_mse[name]
        print(
            f"{name:<12} | "
            f"Train MSE {train_mse[name]:.6f} | "
            f"Val MSE {val_mse[name]:.6f} | "
            f"Best MSE {best_mse[name]:.6f} | "
            f"O MSE {o_mse:.6f} | C MSE {c_mse:.6f} | "
            f"E MSE {e_mse:.6f} | A MSE {a_mse:.6f} | "
            f"N MSE {n_mse:.6f}"
        )

    if no_improve_count >= PATIENCE:
        print(
            f"\nEarly stopping at epoch {epoch + 1} | "
            f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
            f"Patience={no_improve_count}/{PATIENCE}"
        )
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ============================================================
# Final Probe Summary -- MSE only
# ============================================================
print("\n" + "=" * 72)
print("FINAL TIMESFORMER FCAF LINEAR PROBE RESULTS -- UDIVA / MSE")
print("=" * 72)
print(
    f"{'Representation':<18}"
    f"{'Best Epoch':>12}"
    f"{'Best MSE':>16}"
)
print("-" * 72)

for name in probe_names:
    print(
        f"{name:<18}"
        f"{best_epoch[name]:>12d}"
        f"{best_mse[name]:>16.6f}"
    )

print("=" * 72)

del model
del backbone
del optimizer
del train_dataloader
del val_dataloader
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299
Backbone trainable parameters: 0
Probe trainable parameters:    69953
Metric: MSE | Patience: 10


Probe Valid 1/50: 100%|██████████| 191/191 [00:18<00:00, 10.45it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 001 | Mean Val MSE=1.321672 | Best Mean Val MSE=1.321672 | Patience=0/10
block1       | Train MSE 0.071127 | Val MSE 1.425908 | Best MSE 1.425908 | O MSE 1.151810 | C MSE 1.125628 | E MSE 1.333699 | A MSE 1.663431 | N MSE 1.854970
block2       | Train MSE 0.065605 | Val MSE 1.369154 | Best MSE 1.369154 | O MSE 1.172140 | C MSE 1.147999 | E MSE 1.313502 | A MSE 1.544021 | N MSE 1.668111
block3       | Train MSE 0.059187 | Val MSE 1.328866 | Best MSE 1.328866 | O MSE 1.213256 | C MSE 1.098886 | E MSE 1.248144 | A MSE 1.450802 | N MSE 1.633240
block4       | Train MSE 0.053745 | Val MSE 1.316670 | Best MSE 1.316670 | O MSE 1.166806 | C MSE 1.107811 | E MSE 1.251261 | A MSE 1.455708 | N MSE 1.601764
block5       | Train MSE 0.050838 | Val MSE 1.298633 | Best MSE 1.298633 | O MSE 1.070606 | C MSE 1.131727 | E MSE 1.237019 | A MSE 1.418976 | N MSE 1.6348

Probe Valid 2/50: 100%|██████████| 191/191 [00:18<00:00, 10.48it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 002 | Mean Val MSE=1.311471 | Best Mean Val MSE=1.311471 | Patience=0/10
block1       | Train MSE 0.042641 | Val MSE 1.381665 | Best MSE 1.381665 | O MSE 1.275759 | C MSE 1.230350 | E MSE 1.295550 | A MSE 1.538676 | N MSE 1.567990
block2       | Train MSE 0.040149 | Val MSE 1.327746 | Best MSE 1.327746 | O MSE 1.185456 | C MSE 1.188660 | E MSE 1.332085 | A MSE 1.528766 | N MSE 1.403764
block3       | Train MSE 0.037559 | Val MSE 1.345027 | Best MSE 1.328866 | O MSE 1.148344 | C MSE 1.301915 | E MSE 1.337944 | A MSE 1.488361 | N MSE 1.448573
block4       | Train MSE 0.033920 | Val MSE 1.337215 | Best MSE 1.316670 | O MSE 1.121734 | C MSE 1.324710 | E MSE 1.305013 | A MSE 1.512022 | N MSE 1.422597
block5       | Train MSE 0.031475 | Val MSE 1.327260 | Best MSE 1.298633 | O MSE 1.092085 | C MSE 1.309509 | E MSE 1.283471 | A MSE 1.490921 | N MSE 1.4603

...


------------------------------------------------------------------------------------------------------------------------------------
Epoch 020 | Mean Val MSE=1.353270 | Best Mean Val MSE=1.305541 | Patience=8/10
block1       | Train MSE 0.018535 | Val MSE 1.430633 | Best MSE 1.325960 | O MSE 1.157616 | C MSE 1.461811 | E MSE 1.348857 | A MSE 1.604320 | N MSE 1.580561
block2       | Train MSE 0.016375 | Val MSE 1.402419 | Best MSE 1.321950 | O MSE 1.095941 | C MSE 1.401829 | E MSE 1.345166 | A MSE 1.607818 | N MSE 1.561343
block3       | Train MSE 0.014893 | Val MSE 1.385227 | Best MSE 1.315724 | O MSE 1.083250 | C MSE 1.334492 | E MSE 1.308372 | A MSE 1.573282 | N MSE 1.626740
block4       | Train MSE 0.014391 | Val MSE 1.356745 | Best MSE 1.299072 | O MSE 1.065564 | C MSE 1.268071 | E MSE 1.302680 | A MSE 1.546214 | N MSE 1.601199
block5       | Train MSE 0.013609 | Val MSE 1.339894 | Best MSE 1.280817 | O MSE 1.076609 | C MSE 1.247372 | E MSE 1.236923 | A MSE 1.551991 | N MSE 1.5865

Probe Valid 21/50: 100%|██████████| 191/191 [00:45<00:00,  4.19it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 021 | Mean Val MSE=1.333946 | Best Mean Val MSE=1.305541 | Patience=9/10
block1       | Train MSE 0.017784 | Val MSE 1.413536 | Best MSE 1.325960 | O MSE 1.195158 | C MSE 1.343274 | E MSE 1.332417 | A MSE 1.656248 | N MSE 1.540586
block2       | Train MSE 0.015402 | Val MSE 1.398733 | Best MSE 1.321950 | O MSE 1.169840 | C MSE 1.333905 | E MSE 1.269382 | A MSE 1.681625 | N MSE 1.538911
block3       | Train MSE 0.014464 | Val MSE 1.367358 | Best MSE 1.315724 | O MSE 1.126688 | C MSE 1.285945 | E MSE 1.244623 | A MSE 1.586852 | N MSE 1.592683
block4       | Train MSE 0.013721 | Val MSE 1.343307 | Best MSE 1.299072 | O MSE 1.085819 | C MSE 1.230043 | E MSE 1.251260 | A MSE 1.580110 | N MSE 1.569302
block5       | Train MSE 0.012736 | Val MSE 1.336049 | Best MSE 1.280817 | O MSE 1.075454 | C MSE 1.235058 | E MSE 1.228170 | A MSE 1.587838 | N MSE 1.5537

Probe Valid 22/50: 100%|██████████| 191/191 [00:47<00:00,  4.05it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 022 | Mean Val MSE=1.315845 | Best Mean Val MSE=1.305541 | Patience=10/10
block1       | Train MSE 0.018324 | Val MSE 1.376559 | Best MSE 1.325960 | O MSE 1.112495 | C MSE 1.363969 | E MSE 1.291890 | A MSE 1.577867 | N MSE 1.536573
block2       | Train MSE 0.015700 | Val MSE 1.365678 | Best MSE 1.321950 | O MSE 1.106070 | C MSE 1.336252 | E MSE 1.283807 | A MSE 1.572487 | N MSE 1.529772
block3       | Train MSE 0.014170 | Val MSE 1.357817 | Best MSE 1.315724 | O MSE 1.109377 | C MSE 1.291991 | E MSE 1.275411 | A MSE 1.503031 | N MSE 1.609277
block4       | Train MSE 0.013331 | Val MSE 1.333961 | Best MSE 1.299072 | O MSE 1.060114 | C MSE 1.235153 | E MSE 1.301139 | A MSE 1.508716 | N MSE 1.564682
block5       | Train MSE 0.012444 | Val MSE 1.308739 | Best MSE 1.280817 | O MSE 1.053627 | C MSE 1.200525 | E MSE 1.225001 | A MSE 1.525036 | N MSE 1.539